# 🚀 Missão Aurora Siger — Análise de Telemetria

Projeto desenvolvido para a Fase 1 de Ciência da Computação da FIAP.

A análise principal funciona **sem chave de API**. A integração com o Google Gemini é opcional e está separada no final do notebook.


## 1. Instalação e importação das bibliotecas

Execute esta célula primeiro. Ela prepara todas as bibliotecas utilizadas no notebook. **Nenhuma chave de API é necessária nesta etapa.**


In [ ]:
# Instala a biblioteca usada na etapa opcional do Gemini
!pip install -q google-genai

# Imports do projeto
import pandas as pd
import time
from google.colab import files
from google import genai
from getpass import getpass


## 2. Enviar o arquivo CSV

Selecione o arquivo `telemetry_aurora.csv` quando o Colab solicitar.


In [ ]:
uploaded = files.upload()


## 3. Carregar os dados e preparar a análise energética

Esta etapa não utiliza a API do Gemini.


In [ ]:
print('Carregando telemetria da nave Aurora Siger...\n')

df = pd.read_csv("telemetry_aurora.csv")

# Dados de energia
total_capacity = 150           # kWh
estimated_consumption = 40     # kWh
efficiency = 0.90              # 90%

# Consumo total considerando o rendimento
total_energy_consumed = estimated_consumption / efficiency

# Perdas energéticas
energy_losses = total_energy_consumed - estimated_consumption

print('Arquivo carregado com sucesso!')
print(f'Total de registros: {len(df)}')


## 4. Analisar a telemetria

O programa percorre todas as leituras, calcula os dados de energia e verifica os parâmetros de segurança. **Esta etapa funciona sem API.**


In [ ]:
resultados = []

for index, row in df.iterrows():

    print(f'--- Analisando Leitura #{index + 1} ---')

    internal_temperature = row['internal_temperature']
    external_temperature = row['external_temperature']
    structural_integrity = row['structural_integrity']
    energy_level = row['energy_level']
    tank_pressure = row['tank_pressure']
    module_status = row['module_status']

    # Análise energética
    available_energy = (total_capacity * energy_level) / 100
    remaining_energy = available_energy - total_energy_consumed

    print(f'Energia Disponível: {available_energy:.2f} kWh')
    print(f'Consumo estimado (com rendimento): {total_energy_consumed:.2f} kWh')
    print(f'Perdas energéticas: {energy_losses:.2f} kWh')
    print(f'Autonomia restante estimada: {remaining_energy:.2f} kWh\n')

    safe_takeoff = True
    failures = []

    # Parâmetros seguros da missão
    parameters = {
        "Temperatura Interna": 15 <= internal_temperature <= 35,
        "Temperatura Externa": -100 <= external_temperature <= 200,
        "Integridade Estrutural": structural_integrity == 1,
        "Nível de Energia": energy_level >= 80,
        "Pressão dos Tanques": 300 <= tank_pressure <= 500,
        "Status dos Módulos Críticos": module_status == 1
    }

    # Verificação dos parâmetros
    for parameter_name, valid_status in parameters.items():
        if not valid_status:
            safe_takeoff = False
            failures.append(parameter_name)

    # Decisão final
    if safe_takeoff:
        status = "PRONTO PARA DECOLAR"
        print('STATUS: PRONTO PARA DECOLAR\n')
    else:
        status = "DECOLAGEM ABORTADA"
        print('STATUS: DECOLAGEM ABORTADA.\n')
        print('Os Sistemas abaixo falharam:\n')

        for failure_reason in failures:
            print(f' - {failure_reason}')

        print()

    # Armazena o resultado da leitura
    resultado = {
        "leitura": index + 1,
        "internal_temperature": internal_temperature,
        "external_temperature": external_temperature,
        "structural_integrity": structural_integrity,
        "energy_level": energy_level,
        "tank_pressure": tank_pressure,
        "module_status": module_status,
        "available_energy": available_energy,
        "remaining_energy": remaining_energy,
        "safe_takeoff": safe_takeoff,
        "status": status,
        "failures": failures
    }

    resultados.append(resultado)

    print('-' * 70)
    print()

print(f'Varredura completa dos {len(df)} registros finalizada com sucesso.')


## 5. Consultar uma leitura

Altere somente o número da leitura desejada. Esta consulta utiliza os resultados já salvos e não chama a API.


In [ ]:
numero_leitura = 1

resultado = resultados[numero_leitura - 1]

print(f'Leitura: {resultado["leitura"]}')
print(f'Temperatura interna: {resultado["internal_temperature"]} °C')
print(f'Temperatura externa: {resultado["external_temperature"]} °C')
print(f'Nível de energia: {resultado["energy_level"]}%')
print(f'Pressão dos tanques: {resultado["tank_pressure"]} kPa')
print(f'Energia disponível: {resultado["available_energy"]:.2f} kWh')
print(f'Autonomia restante: {resultado["remaining_energy"]:.2f} kWh')
print(f'Status: {resultado["status"]}')
print(f'Falhas: {resultado["failures"]}')


## 6. Mostrar os resultados em tabela

A tabela reúne todas as leituras processadas. Esta etapa também funciona sem API.


In [ ]:
tabela_resultados = pd.DataFrame(resultados)

tabela_resultados


---

# Etapa opcional — Google Gemini

As células abaixo só precisam ser executadas se você possuir uma chave da API do Gemini. A decisão de decolagem já foi realizada pelas regras de segurança do código nas etapas anteriores.


## 7. Configurar a API do Gemini

A chave é digitada de forma oculta e não deve ser salva diretamente no notebook ou no repositório.


In [ ]:
GEMINI_API_KEY = getpass("Cole sua Gemini API Key: ")
client = genai.Client(api_key=GEMINI_API_KEY)

def analyze_with_gemini(data):
    prompt = f"""
Você é o sistema analítico de suporte à missão da nave Aurora Siger.
Analise esta leitura de telemetria que apresentou falha no pré-lançamento:

- Temperatura Interna: {data['internal_temperature']} °C (Ideal: 15 a 35)
- Temperatura Externa: {data['external_temperature']} °C (Ideal: -100 a 200)
- Integridade Estrutural: {data['structural_integrity']} (Exigido: 1)
- Nível de Energia: {data['energy_level']}% (Mínimo: 80)
- Pressão dos Tanques: {data['tank_pressure']} kPa (Ideal: 300 a 500)
- Status dos Módulos Críticos: {data['module_status']} (Exigido: 1)

Forneça uma resposta objetiva estruturada exatamente nos seguintes tópicos:
CLASSIFICAÇÃO: (ATENÇÃO ou CRÍTICO)
ANOMALIAS: (Descreva o que causou o problema)
RISCOS: (Impacto estrutural ou operacional)
RECOMENDAÇÃO: (Ajuste corretivo de engenharia)
"""

    response = client.models.generate_content(
        model="gemini-3.5-flash-lite",
        contents=prompt
    )

    return response.text

print("Gemini configurado com sucesso!")


## 8. Analisar com Gemini somente as leituras com falha

O Gemini é acionado apenas para registros que já receberam o status **DECOLAGEM ABORTADA** pelas verificações do código.


In [ ]:
for resultado in resultados:

    if resultado["safe_takeoff"] == False:

        print(f'--- Gemini analisando Leitura #{resultado["leitura"]} ---\n')

        failure_data = {
            'internal_temperature': resultado['internal_temperature'],
            'external_temperature': resultado['external_temperature'],
            'structural_integrity': resultado['structural_integrity'],
            'energy_level': resultado['energy_level'],
            'tank_pressure': resultado['tank_pressure'],
            'module_status': resultado['module_status']
        }

        try:
            print('[IA] Acionando triagem assistida do Gemini...\n')

            ai_analysis = analyze_with_gemini(failure_data)
            resultado["analise_ia"] = ai_analysis

            print(ai_analysis)

        except Exception as e:
            print(f'Aviso de conexão com a API: {e}')

        print('\n' + '=' * 100 + '\n')
        time.sleep(1)


## 9. Ver uma resposta do Gemini já salva

Depois que a análise com Gemini for executada uma vez, esta célula apenas mostra a resposta armazenada e não realiza uma nova chamada.


In [ ]:
numero_leitura = 3

resultado = resultados[numero_leitura - 1]

if "analise_ia" in resultado:
    print(resultado["analise_ia"])
else:
    print("Esta leitura ainda não possui uma análise do Gemini salva.")


## 10. Atualizar a tabela após a análise com IA

Execute novamente esta célula se quiser visualizar a coluna `analise_ia` após usar o Gemini.


In [ ]:
tabela_resultados = pd.DataFrame(resultados)

tabela_resultados
